In [1]:
import json

In [2]:
#!/usr/bin/env python3
import argparse
import hashlib
from pathlib import Path
from itertools import combinations

import pandas as pd


# -----------------------------
# Helpers
# -----------------------------
def stable_hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def canon_labelset(x):
    """
    Make labels hashable + stable:
      - list -> sorted unique tuple
      - scalar -> string
    """
    if isinstance(x, list):
        return tuple(sorted(set(map(str, x))))
    # sometimes it's a string, numpy array, etc.
    return tuple([str(x)])

def ensure_labels_column(df: pd.DataFrame, labels_col_guess=("labels", "label", "ttp", "ttps")) -> pd.DataFrame:
    """
    Tries to ensure there's a 'labels' column containing a list-like label set.
    Adjust this if your JSON schema differs.
    """
    df = df.copy()

    if "labels" in df.columns:
        return df

    for c in labels_col_guess:
        if c in df.columns:
            # If it's a single label, convert to list
            if df[c].apply(lambda v: isinstance(v, list)).any():
                df["labels"] = df[c]
            else:
                df["labels"] = df[c].apply(lambda v: [v] if pd.notna(v) else [])
            return df

    raise ValueError(f"Could not find a labels column. Available columns: {df.columns.tolist()}")

def ensure_sentence_column(df: pd.DataFrame, sentence_col_guess=("sentence", "text", "utterance", "content")) -> pd.DataFrame:
    df = df.copy()
    if "sentence" in df.columns:
        return df
    for c in sentence_col_guess:
        if c in df.columns:
            df["sentence"] = df[c]
            return df
    raise ValueError(f"Could not find a sentence column. Available columns: {df.columns.tolist()}")

def build_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["labels_key"] = df["labels"].apply(canon_labelset)
    # Key uses both sentence and label-set; include a delimiter to avoid accidental collisions
    df["key"] = (df["sentence"].astype(str) + "||" + df["labels_key"].apply(lambda t: ",".join(t))).apply(stable_hash)
    return df


# -----------------------------
# Main overlap logic
# -----------------------------
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_json(path)
    df = ensure_sentence_column(df)
    df = ensure_labels_column(df)
    return df

def summarize_dataset(name: str, df: pd.DataFrame):
    n = len(df)
    n_unique = df["key"].nunique()
    n_dups = n - n_unique
    print(f"\n=== {name} ===")
    print(f"Rows: {n}")
    print(f"Unique (sentence+labelset) keys: {n_unique}")
    print(f"Within-dataset duplicates: {n_dups} ({(n_dups / max(n,1))*100:.2f}%)")

def pairwise_overlap(name_a, df_a, name_b, df_b):
    keys_a = set(df_a["key"])
    keys_b = set(df_b["key"])
    inter = keys_a & keys_b
    return inter

def print_overlap_examples(df_a, df_b, overlap_keys, max_examples=5):
    if not overlap_keys:
        return
    # show examples using rows from A (and optionally B)
    ex = df_a[df_a["key"].isin(list(overlap_keys))].head(max_examples)
    for _, r in ex.iterrows():
        sent = str(r["sentence"])
        labels = r["labels_key"]
        print("-" * 80)
        print(f"Sentence: {sent[:400]}")
        print(f"Labels:   {labels}")

In [14]:
def main():
    # --------------------------------------------------
    # CONFIG — EDIT THESE
    # --------------------------------------------------
    DATASETS = {
        # TRAM
        "TRAM_ORIG":  "datasets/tram_train.json",
        "TRAM_HIER":  "datasets/tram_augmented_hierarchy_embeddings.json",
        "TRAM_MITRE": "datasets/tram_train_augmented_mitre.json",

        # BOSCH
        # "BOSCH_ORIG":  "datasets/bosch_train.json",
        # "BOSCH_HIER":  "datasets/bosch_augmented_hierarchy.json",
        # "BOSCH_MITRE": "datasets/bosch_train_augmented_mitre.json",
    }

    TOP_K_PAIRS = 5        # print examples for top-N overlapping pairs
    EXAMPLES_PER_PAIR = 5 # number of example sentences per pair

    # --------------------------------------------------
    # Load all datasets
    # --------------------------------------------------
    datasets = {}
    for name, path in DATASETS.items():
        path = Path(path).expanduser()
        if not path.exists():
            raise FileNotFoundError(path)

        df = load_dataset(path)
        df = build_keys(df)
        datasets[name] = df

    # --------------------------------------------------
    # Summaries
    # --------------------------------------------------
    print("\n############################")
    print("# Dataset summaries")
    print("############################")
    for name, df in datasets.items():
        summarize_dataset(name, df)

    # --------------------------------------------------
    # Pairwise overlaps
    # --------------------------------------------------
    print("\n############################")
    print("# Pairwise overlaps")
    print("############################")

    rows = []
    names = list(datasets.keys())
    for a, b in combinations(names, 2):
        df_a, df_b = datasets[a], datasets[b]
        inter = pairwise_overlap(a, df_a, b, df_b)
        overlap = len(inter)
        pct_a = overlap / max(df_a["key"].nunique(), 1) * 100
        pct_b = overlap / max(df_b["key"].nunique(), 1) * 100
        rows.append((a, b, overlap, pct_a, pct_b, inter))

    rows_sorted = sorted(rows, key=lambda x: x[2], reverse=True)

    if rows_sorted:
        table = pd.DataFrame(
            [(a, b, overlap, f"{pct_a:.2f}%", f"{pct_b:.2f}%")
             for a, b, overlap, pct_a, pct_b, _ in rows_sorted],
            columns=["A", "B", "Overlap (count)", "Overlap % of A", "Overlap % of B"],
        )
        print(table.to_string(index=False))
    else:
        print("Need at least two datasets to compute pairwise overlap.")
        return

    # --------------------------------------------------
    # Example overlaps
    # --------------------------------------------------
    print("\n############################")
    print("# Example overlaps (top pairs)")
    print("############################")

    for i, (a, b, overlap, pct_a, pct_b, inter) in enumerate(rows_sorted[:TOP_K_PAIRS], start=1):
        if overlap == 0:
            break
        print(f"\n[{i}] {a} vs {b} | overlap={overlap} | {pct_a:.2f}% of {a} | {pct_b:.2f}% of {b}")
        print_overlap_examples(
            datasets[a],
            datasets[b],
            inter,
            max_examples=EXAMPLES_PER_PAIR,
        )


if __name__ == "__main__":
    main()



############################
# Dataset summaries
############################

=== TRAM_ORIG ===
Rows: 15358
Unique (sentence+labelset) keys: 15021
Within-dataset duplicates: 337 (2.19%)

=== TRAM_HIER ===
Rows: 4506
Unique (sentence+labelset) keys: 4506
Within-dataset duplicates: 0 (0.00%)

=== TRAM_MITRE ===
Rows: 7929
Unique (sentence+labelset) keys: 7929
Within-dataset duplicates: 0 (0.00%)

############################
# Pairwise overlaps
############################
        A          B  Overlap (count) Overlap % of A Overlap % of B
TRAM_ORIG  TRAM_HIER                0          0.00%          0.00%
TRAM_ORIG TRAM_MITRE                0          0.00%          0.00%
TRAM_HIER TRAM_MITRE                0          0.00%          0.00%

############################
# Example overlaps (top pairs)
############################


In [13]:
#!/usr/bin/env python3
from pathlib import Path
import hashlib
import pandas as pd

# -----------------------------
# Config
# -----------------------------
TRAM_HIER_PATH  = Path("datasets/tram_augmented_hierarchy_embeddings.json")
TRAM_MITRE_PATH = Path("datasets/tram_train_augmented_mitre.json")

OUT_HIER_CLEAN_PATH = Path("datasets/tram_augmented_hierarchy_embeddings.json")

# -----------------------------
# Helpers
# -----------------------------
def stable_hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def canon_labelset(x):
    if isinstance(x, list):
        return tuple(sorted(set(map(str, x))))
    return tuple([str(x)])

def ensure_sentence_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "sentence" not in df.columns:
        raise ValueError(f"Missing 'sentence'. Columns: {df.columns.tolist()}")
    if "labels" not in df.columns:
        # fall back if your file uses 'label'
        if "label" in df.columns:
            # make it list-like
            df["labels"] = df["label"].apply(lambda v: v if isinstance(v, list) else [v])
        else:
            raise ValueError(f"Missing 'labels' (and no 'label'). Columns: {df.columns.tolist()}")
    return df

def add_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["labels_key"] = df["labels"].apply(canon_labelset)
    df["key"] = (df["sentence"].astype(str) + "||" + df["labels_key"].apply(lambda t: ",".join(t))).apply(stable_hash)
    return df

# -----------------------------
# Main
# -----------------------------
def main():
    # Load
    hier = pd.read_json(TRAM_HIER_PATH)
    mitre = pd.read_json(TRAM_MITRE_PATH)

    hier = ensure_sentence_labels(hier)
    mitre = ensure_sentence_labels(mitre)

    # Keys
    hier = add_keys(hier)
    mitre = add_keys(mitre)

    print("\n=== BEFORE ===")
    print("TRAM_HIER rows:", len(hier))
    print("TRAM_HIER unique keys:", hier["key"].nunique())
    print("TRAM_MITRE rows:", len(mitre))
    print("TRAM_MITRE unique keys:", mitre["key"].nunique())

    # 1) Remove internal duplicates in TRAM_HIER
    before_internal = len(hier)
    hier = hier.drop_duplicates(subset=["key"], keep="first")
    removed_internal = before_internal - len(hier)

    # 2) Remove overlap with TRAM_MITRE from TRAM_HIER
    mitre_keys = set(mitre["key"].unique())
    before_overlap = len(hier)
    overlap_mask = hier["key"].isin(mitre_keys)
    overlap_count = int(overlap_mask.sum())
    hier_clean = hier.loc[~overlap_mask].copy()
    removed_overlap = before_overlap - len(hier_clean)

    print("\n=== REMOVALS ===")
    print(f"Removed internal dups from TRAM_HIER: {removed_internal}")
    print(f"Removed overlap (TRAM_HIER ∩ TRAM_MITRE) from TRAM_HIER: {overlap_count}")

    print("\n=== AFTER ===")
    print("TRAM_HIER cleaned rows:", len(hier_clean))
    print("TRAM_HIER cleaned unique keys:", hier_clean['key'].nunique())

    # Optional sanity check: ensure no overlap remains
    remaining_overlap = len(set(hier_clean["key"]) & mitre_keys)
    print("Remaining overlap with TRAM_MITRE:", remaining_overlap)

    # Save (drop helper cols)
    drop_cols = [c for c in ["labels_key", "key"] if c in hier_clean.columns]
    hier_clean.drop(columns=drop_cols, inplace=True)

    OUT_HIER_CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    hier_clean.to_json(OUT_HIER_CLEAN_PATH, orient="columns", indent=2)

    print(f"\nSaved cleaned TRAM_HIER to: {OUT_HIER_CLEAN_PATH}")

if __name__ == "__main__":
    main()



=== BEFORE ===
TRAM_HIER rows: 8180
TRAM_HIER unique keys: 7985
TRAM_MITRE rows: 7929
TRAM_MITRE unique keys: 7929

=== REMOVALS ===
Removed internal dups from TRAM_HIER: 195
Removed overlap (TRAM_HIER ∩ TRAM_MITRE) from TRAM_HIER: 3479

=== AFTER ===
TRAM_HIER cleaned rows: 4506
TRAM_HIER cleaned unique keys: 4506
Remaining overlap with TRAM_MITRE: 0

Saved cleaned TRAM_HIER to: datasets/tram_augmented_hierarchy_embeddings.json
